# Notebook 2 — Full experiment

Runs the **complete** intended experiment by importing and reusing the corrected `thesis_pipeline` functions (no duplicated logic).

**Safe for long runs**
- Every stage is checkpointed; re-running resumes instead of repeating work.
- A single failing case is logged (status + error) and the run continues.
- API/parse failures are separated from genuine wrong answers (`accuracy_successful_only`, `failure_rate`).
- Reproducible via a fixed `GLOBAL_SEED`; API keys are never printed.

> Run **01_RUN_100_EXAMPLES_TEST.ipynb** first and confirm it is healthy.

In [ ]:
# --- Environment & configuration -------------------------------------------
import os, sys, json, importlib
from pathlib import Path
import pandas as pd

# Make the project importable whether the kernel starts in notebooks/ or root.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
for _m in [m for m in list(sys.modules) if m == "thesis_pipeline" or m.startswith("thesis_pipeline.")]:
    del sys.modules[_m]
importlib.invalidate_caches()

from dotenv import load_dotenv
load_dotenv(ROOT / ".env")

# TLS: the SDKs verify certificates by default. Behind a TLS-intercepting
# corporate proxy every API call raises APIConnectionError and the pipeline
# would record empty predictions. Setting VERIFY_SSL=false in .env (or here)
# restores connectivity. Leave it "true" on a normal network.
os.environ.setdefault("VERIFY_SSL", os.environ.get("VERIFY_SSL", "true"))

from thesis_pipeline import initialize_thesis
from thesis_pipeline.evaluation import score, metrics
from thesis_pipeline.checkpoint import load_completed

# Report key presence ONLY (never print the key values themselves).
print("OPENAI_API_KEY set:  ", bool(os.environ.get("OPENAI_API_KEY")))
print("ANTHROPIC_API_KEY set:", bool(os.environ.get("ANTHROPIC_API_KEY")))
print("VERIFY_SSL:          ", os.environ.get("VERIFY_SSL"))

In [ ]:
from thesis_pipeline import (run_single_baselines, run_agreement_experiments,
                             select_best_agreement_method, run_final_comparison,
                             build_thesis_outputs)

In [ ]:
# --- Dataset & full-experiment configuration -------------------------------
DATASET = {
    "name": "final_dataset_1",
    # Reliable extracted gold letters (built by tools/build_gold_labels.py).
    "path": "data/final_dataset_1_with_gold.csv",
    "columns": {"instruction": "instruction", "input": "input", "gold": "gold_letter"},
}
GLOBAL_SEED = 42

# RUN_MODE "full" scores every eligible case. THESIS_MAX_EXAMPLES can cap the
# test split for a shorter run; leave it unset for the complete experiment.
_cap = os.environ.get("THESIS_MAX_EXAMPLES")
RUN_MODE = "test" if _cap else "full"
extra = {"test_size": int(_cap)} if _cap else {}

pipeline = initialize_thesis(
    DATASET, RUN_MODE=RUN_MODE, GLOBAL_SEED=GLOBAL_SEED, root=ROOT,
    mock=False, output_dir="results/final_dataset_1", **extra,
)
v = pipeline.bundle.validation
print("Dataset:", pipeline.bundle.dataset_name, "| RUN_MODE:", RUN_MODE)
print(f"Eligible: {v['eligible_cases']} | excluded: {v['excluded_cases']}")
print(f"Development: {len(pipeline.bundle.cases('development'))} | Test: {len(pipeline.bundle.cases('test'))}")

### 1. Single-model baselines

In [ ]:
# --- 1) Single-model baselines (TEST split, checkpointed) ------------------
# Reuses the corrected pipeline. Every case that fails is recorded with its
# status/error and retried on resume; the run does not crash on a single error.
single_results = run_single_baselines()
single_results[["system","split","n","successful_n","failure_rate",
                "accuracy","accuracy_successful_only","precision","recall",
                "macro_f1","critical_safety_error_rate"]]

### 2. Development-only agreement-method comparison

In [ ]:
# --- 2) Agreement-method comparison (DEVELOPMENT split only) ---------------
agreement_results = run_agreement_experiments()
agreement_results[["rank","method","split","n","successful_n","failure_rate",
                   "accuracy","precision","recall","macro_f1",
                   "critical_safety_error_rate","selected"]]

### 3. Freeze the selected method

In [ ]:
# --- 3) Freeze the winning agreement method --------------------------------
selection = select_best_agreement_method()
print("Frozen agreement method:", selection["selected_method"])
selection

### 4. Final held-out comparison

In [ ]:
# --- 4) Final held-out comparison (TEST split) -----------------------------
final_results = run_final_comparison()
final_table = (pd.DataFrame.from_dict(final_results["systems"], orient="index")
               .rename_axis("system").reset_index())
final_table[["system","n","successful_n","failure_rate","accuracy",
             "accuracy_successful_only","macro_f1","critical_safety_error_rate"]]

### 5. Reports & figures

In [ ]:
# --- 5) Build thesis reports & figures -------------------------------------
outputs = build_thesis_outputs()
print("Reports:");  print(*outputs["reports"], sep="\n")
print("Figures:");  print(*outputs["figures"], sep="\n")